rainfall_analysis
# County Rainfall and Planting Advisory
Cleaning hand-typed rainfall data and finding planting windows for five Kenyan counties with NumPy.

In [1]:
import csv
import numpy as np

months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

In [2]:
def clean_readings(raw):
    """Convert raw text to ints; impute invalid values with the rounded mean of the valid ones."""
    cleaned = []
    invalid_idx = []
    for i, value in enumerate(raw):
        try:
            cleaned.append(int(value))      # int() tolerates spaces, e.g. " 20 "
        except ValueError:                  # "N/A" and "" both raise ValueError
            cleaned.append(None)
            invalid_idx.append(i)

    valid = [v for v in cleaned if v is not None]
    fill = round(sum(valid) / len(valid))

    for i in invalid_idx:
        cleaned[i] = fill
    return cleaned, invalid_idx, fill


machakos_clean, imputed_idx, fill_value = clean_readings(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print("Imputation value (rounded mean of valid readings):", fill_value)
print("Imputed months:", [months[i] for i in imputed_idx])

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputation value (rounded mean of valid readings): 68
Imputed months: ['April', 'September']


In [3]:
rain = np.array(clean_rows + [machakos_clean])

print(rain)
print("Shape:", rain.shape)
print("dtype:", rain.dtype)

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
dtype: int64


In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county:<9} annual total: {total} mm")

print()
for month, avg in zip(months, monthly_avg):
    print(f"{month:<10} average across counties: {avg:.1f} mm")

Nairobi   annual total: 1045 mm
Nakuru    annual total: 916 mm
Kisumu    annual total: 1371 mm
Mombasa   annual total: 1075 mm
Machakos  annual total: 811 mm

January    average across counties: 47.0 mm
February   average across counties: 47.6 mm
March      average across counties: 91.0 mm
April      average across counties: 164.4 mm
May        average across counties: 171.2 mm
June       average across counties: 67.6 mm
July       average across counties: 56.6 mm
August     average across counties: 58.2 mm
September  average across counties: 64.2 mm
October    average across counties: 67.8 mm
November   average across counties: 127.4 mm
December   average across counties: 80.6 mm


In [5]:
wettest_idx = np.argmax(rain, axis=1)

for county, idx in zip(counties, wettest_idx):
    print(f"{county:<9} wettest month: {months[idx]}")

Nairobi   wettest month: April
Nakuru    wettest month: April
Kisumu    wettest month: April
Mombasa   wettest month: May
Machakos  wettest month: November


In [6]:
dry_mask = rain < 50
dry_counts = dry_mask.sum(axis=1)

for county, n in zip(counties, dry_counts):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


In [7]:
pair_sums = rain[:, :-1] + rain[:, 1:]
best_start = np.argmax(pair_sums, axis=1)
best_total = pair_sums[np.arange(len(counties)), best_start]

windows = [f"{months[s]}-{months[s + 1]}" for s in best_start]

for county, w, t in zip(counties, windows, best_total):
    print(f"{county:<9} planting window: {w} ({t} mm combined)")

Nairobi   planting window: April-May (395 mm combined)
Nakuru    planting window: April-May (265 mm combined)
Kisumu    planting window: April-May (420 mm combined)
Mombasa   planting window: April-May (390 mm combined)
Machakos  planting window: November-December (278 mm combined)


In [8]:
wettest_names = [months[i] for i in wettest_idx]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for row in zip(counties, annual_totals.tolist(), wettest_names, dry_counts.tolist(), windows):
        writer.writerow(row)

with open("rainfall_summary.csv") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December



## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should be advised to plant during April-May, when each of these counties receives its highest two-month rainfall (265 mm in Nakuru, 390-420 mm in the other three). Machakos follows a different pattern, with its best window in November-December, so SMS advice sent there should not copy the April-May message used elsewhere. Machakos is also the driest county, with 6 months below 50 mm and an annual total of 811 mm, so drought-tolerant crops and water-saving practices deserve emphasis. Kisumu is the wettest county (1,371 mm) and has no dry months, making it the lowest-risk location for rain-fed planting. **Caveat:** the Machakos row was hand-typed and two of its twelve readings (April and September) were missing or invalid and were replaced with the mean of the valid values, so the Machakos annual total and dry-month count are less reliable than those of the other counties and should be confirmed against the station records.

## Data Quality Notes

- **Invalid entries:** the Machakos row contained "N/A" (April) and an empty string (September). Both raised `ValueError` when converted with `int()` and were treated as missing.
- **Whitespace:** " 20 " (June) had stray spaces. `int()` handles this, so it was kept as a valid reading of 20 mm.
- **Imputation:** the two missing months were filled with 68 mm, the rounded mean of the ten valid Machakos readings. This is a placeholder, not a real measurement.
- **Impact:** the imputed values add 136 mm to Machakos' annual total but do not change its November-December planting window or its dry-month count.
- **Other counties:** the four station rows were supplied clean and used as given. One year of data per county also cannot show year-to-year variation.